# Road damage detection: train YOLOv8m (Kaggle or Colab)

Trains **YOLOv8m** on RDD2022 to detect 4 road-damage classes: D00 longitudinal crack, D10
transverse crack, D20 alligator crack and D40 pothole. It then evaluates the model on the val and test
splits. All code is in this notebook, and it detects whether it runs on Kaggle or Google Colab.

**Expected time on a T4 GPU:** ~22 min per epoch, ~11 h (one or two sessions) for 30 epochs.

### Run on Kaggle (recommended: 30 free GPU hours per week, runs in the background)
1. On kaggle.com: *Create → New Notebook*, then *File → Import Notebook* and paste this notebook's
   GitHub URL (or upload the `.ipynb`).
2. In the right-hand *Settings* panel, set *Accelerator* to **GPU T4 x2** and turn **Internet on**.
   Internet needs a phone-verified Kaggle account. Don't pick P100: current PyTorch no longer
   supports it.
3. Optional, but saves a download: *Add Input* → search **rdd-2022** (by aliabdelmenam) → add it.
4. *Save Version → **Save & Run All (Commit)***. It runs in the background for up to 12 h, so you
   can close the browser. Results appear in the version's *Output* tab: `runs/`, plus
   `results/metrics.csv` once training has finished.
5. **If the run stopped at `MAX_HOURS`** (it prints "Paused"), open the notebook again and use
   *Add Input → Your Work → Notebook Output* to add **this notebook's previous version**. Then
   *Save & Run All* again. The previous checkpoint is found automatically and training continues.

### Run on Google Colab
*Runtime → Change runtime type → T4 GPU*, then *Run all*, and allow Google Drive access.
Checkpoints go to Google Drive (`DRIVE_DIR`). After a disconnect, *Run all* again to continue.
The free Colab GPU allowance is only a few hours per day.

### Data fixes applied automatically
The Kaggle RDD2022 export has 5 class ids, where 3 is "other corruption" and 4 is the pothole. It
also uses a random split, so neighbouring video frames leak between train and test. The data cell
drops "other corruption", maps 4 → D40, and re-splits by 50-frame blocks per country (seed 42, about
70/15/15). This is the same split as the local runs. Data that is already fixed is used as it is.


## Settings

In [ ]:
# ================= Settings =================
MODEL_NAME = "yolov8m"            # runs are named <MODEL_NAME>_<IMGSZ>_s<SEED>
WEIGHTS = "yolov8m.pt"           # COCO-pretrained weights, downloaded automatically
EPOCHS = 30                     # use the same value for all 4 models so they are comparable
IMGSZ = 640
BATCH = 16                      # fits a 16 GB T4; lower it if you get "CUDA out of memory"
SEED = 42
MAX_HOURS = 10.5                # stop cleanly before the 12 h session limit, then run again to continue
CACHE = False                   # "disk" speeds up epochs on Colab's 2 CPUs (needs ~25 GB of Colab disk)

# Data: set ONE of these two
KAGGLE_DATASET = "aliabdelmenam/rdd-2022"   # public RDD2022 YOLO export (11 GB)
DATA_PATH = ""                  # e.g. "/content/drive/MyDrive/road-damage/data.zip"

DRIVE_DIR = "/content/drive/MyDrive/road-damage"   # Colab: checkpoints + results go here (Kaggle uses /kaggle/working)
RESUME_FROM = ""                # folder holding an earlier runs/<name> to continue (Kaggle finds it in the inputs)

## Setup
Detects Kaggle / Colab, mounts Google Drive on Colab, picks up a previous run, and installs packages.

In [ ]:
import csv, glob, json, math, os, random, re, shutil, subprocess, sys, time
from collections import Counter, defaultdict
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = Path("/kaggle/working").is_dir()
KAGGLE_INPUT = Path("/kaggle/input")
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LOCAL = Path("/content")                  # fast local disk for the dataset
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):  # optional Colab secrets for private Kaggle datasets
        try:
            from google.colab import userdata
            os.environ.setdefault(name, userdata.get(name))
        except Exception:
            pass
elif ON_KAGGLE:                               # dataset in /tmp; /kaggle/working is saved as the version's output
    LOCAL = Path("/tmp")
    DRIVE_DIR = "/kaggle/working"
else:                                         # elsewhere (e.g. local Jupyter): keep everything here
    LOCAL = Path.cwd() / "colab_local"
    if DRIVE_DIR.startswith("/content/"):
        DRIVE_DIR = str(LOCAL / "drive")

RUNS = Path(DRIVE_DIR) / "runs"
RESULTS = Path(DRIVE_DIR) / "results"
for d in (LOCAL, RUNS, RESULTS):
    d.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"{MODEL_NAME}_{IMGSZ}_s{SEED}"
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
WORKERS = min(8, os.cpu_count() or 2)
random.seed(SEED)


def run_progress(run):
    """How far a run folder got: finished runs first, then by number of epochs logged."""
    if (run / "DONE").exists() or (run / "model.pt").exists():
        return 10**6
    log = run / "results.csv" if (run / "results.csv").exists() else run / "metrics.csv"
    return sum(1 for _ in open(log)) if log.exists() else 0


def find_previous_run(roots, max_depth=5):
    """The most advanced runs/<RUN_NAME> folder below roots (e.g. a previous Kaggle version added as input)."""
    found = []
    for root in map(Path, roots):
        for dirpath, dirnames, _ in os.walk(root):
            here = Path(dirpath)
            if here.name == RUN_NAME and here.parent.name == "runs":
                found.append(here)
                dirnames[:] = []
                continue
            depth = len(here.relative_to(root).parts)
            dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if d not in ("images", "labels")]
    return max(found, key=run_progress) if found else None


# Continue a previous run: Kaggle finds it among the inputs, anywhere else set RESUME_FROM
if not (RUNS / RUN_NAME).exists() and (RESUME_FROM or ON_KAGGLE):
    previous = find_previous_run([RESUME_FROM] if RESUME_FROM else [KAGGLE_INPUT])
    if previous:
        shutil.copytree(previous, RUNS / RUN_NAME)
        if (previous.parent.parent / "results").is_dir():
            shutil.copytree(previous.parent.parent / "results", RESULTS, dirs_exist_ok=True)
        for p in [RUNS / RUN_NAME, *(RUNS / RUN_NAME).rglob("*"), *RESULTS.rglob("*")]:
            p.chmod(0o755 if p.is_dir() else 0o644)     # inputs are read-only; training must overwrite them
        print(f"Continuing the previous run from {previous} ({run_progress(previous)} log lines)")

platform = "Colab" if ON_COLAB else "Kaggle" if ON_KAGGLE else "local"
print(f"Platform: {platform} | run: {RUN_NAME} | outputs: {DRIVE_DIR}")

In [ ]:
pkgs = ['ultralytics>=8.3', 'kagglehub']
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError("pip install failed. On Kaggle turn on Settings → Internet (needs a phone-verified "
                       "account).\n" + result.stderr[-2000:])
print("Installed:", ", ".join(pkgs))

In [ ]:
import numpy as np
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB")
    arch, built_for = f"sm_{props.major}{props.minor}", torch.cuda.get_arch_list()
    if built_for and arch not in built_for:
        raise RuntimeError(f"This PyTorch build does not support the {props.name} ({arch}). "
                           "On Kaggle choose Settings → Accelerator → GPU T4 x2.")
    DEVICE = 0
else:
    msg = ("No GPU. Either the free GPU quota is used up (Colab: a few hours per day; Kaggle: 30 h per week), "
           "or the session has no GPU (Colab: Runtime → Change runtime type → T4 GPU; "
           "Kaggle: Settings → Accelerator → GPU T4 x2).")
    if ON_COLAB or ON_KAGGLE:
        raise RuntimeError(msg + " Stopping here so nothing trains on CPU.")
    print(msg)
    DEVICE = "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)

## Data
Gets the dataset (from Kaggle or Drive), then writes a fixed copy to Colab's local disk: 4 classes and a
leak-free split by frame blocks. Images are symlinked, not copied. This takes a few minutes once per
Colab session.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLITS = ("train", "val", "test")
CLASS_REMAP = {0: 0, 1: 1, 2: 2, 4: 3}       # source id -> project id; id 3 ("other corruption") is dropped
DATA = LOCAL / "rdd2022"                      # prepared dataset
DATA_YAML = LOCAL / "rdd2022.yaml"


def find_dataset(root, max_depth=6):
    """First folder below root that has train/images and val/images."""
    root = Path(root)
    for dirpath, dirnames, _ in os.walk(root):
        here = Path(dirpath)
        if (here / "train" / "images").is_dir() and (here / "val" / "images").is_dir():
            return here
        depth = len(here.relative_to(root).parts)
        dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if d not in ("images", "labels")]
    return None


def get_raw_dataset():
    if ON_KAGGLE:                                 # dataset already attached with "Add Input"?
        attached = find_dataset(KAGGLE_INPUT)
        if attached is not None:
            return attached
    if KAGGLE_DATASET:
        import kagglehub
        path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    elif DATA_PATH:
        path = Path(DATA_PATH)
        if path.suffix == ".zip":
            raw = LOCAL / "raw"
            if not raw.exists():
                local_zip = LOCAL / path.name
                if not local_zip.exists():
                    print("Copying the zip from Drive ...")
                    shutil.copy(path, local_zip)          # unzipping straight from Drive is much slower
                print("Unzipping ...")
                shutil.unpack_archive(str(local_zip), str(raw))
                local_zip.unlink()
            path = raw
    else:
        raise ValueError("Set KAGGLE_DATASET or DATA_PATH in the Settings cell.")
    found = find_dataset(path)
    if found is None:
        raise FileNotFoundError(f"No folder with train/images and val/images found under {path}")
    return found


def country_and_index(stem):
    m = re.match(r"(.+?)_(\d+)$", stem)
    return (m.group(1), int(m.group(2))) if m else (stem, None)


def has_class_id(src, class_id):
    prefix = f"{class_id} "
    for f in src.glob("*/labels/*.txt"):
        with open(f, encoding="utf-8") as fh:
            if any(line.startswith(prefix) for line in fh):
                return True
    return False


def block_split(items, block_size=50, val_frac=0.15, test_frac=0.15, seed=42):
    """Assign whole blocks of consecutive frames (per country) to one split, stratified by country."""
    groups = defaultdict(list)
    for item in items:
        country, idx = country_and_index(item[0])
        groups[(country, idx // block_size if idx is not None else item[0])].append(item)
    rng = random.Random(seed)
    by_country = defaultdict(list)
    for key in groups:
        by_country[key[0]].append(key)
    target = {}
    for country, keys in sorted(by_country.items()):
        keys = sorted(keys, key=str)
        rng.shuffle(keys)
        n = len(keys)
        n_test = max(1, round(n * test_frac)) if n >= 3 else 0
        n_val = max(1, round(n * val_frac)) if n >= 3 else 0
        for i, key in enumerate(keys):
            target[key] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return {stem: target[key] for key, members in groups.items() for stem, _, _ in members}


def prepare_dataset(src, dst):
    items = [(p.stem, s, p) for s in SPLITS if (src / s / "images").is_dir()
             for p in (src / s / "images").iterdir() if p.suffix.lower() in IMG_EXTS]
    already_fixed = (src / "splits.csv").exists() or (src / ".fixed_by_fix_prepared_data").exists()
    remap = not already_fixed and has_class_id(src, 4)
    split_of = {stem: s for stem, s, _ in items} if already_fixed else block_split(items)
    print(f"{len(items)} images | already fixed: {already_fixed} | remap classes: {remap}")

    for s in SPLITS:
        (dst / s / "images").mkdir(parents=True, exist_ok=True)
        (dst / s / "labels").mkdir(parents=True, exist_ok=True)
    counts, dropped, use_links = {s: Counter() for s in SPLITS}, Counter(), True
    for stem, s, img in items:
        out = split_of[stem]
        label = src / s / "labels" / f"{stem}.txt"
        lines = label.read_text(encoding="utf-8").split("\n") if label.exists() else []
        kept = []
        for line in lines:
            fields = line.split()
            if not fields:
                continue
            cid = int(float(fields[0]))
            if remap:
                if cid not in CLASS_REMAP:
                    dropped[cid] += 1
                    continue
                cid = CLASS_REMAP[cid]
            kept.append(" ".join([str(cid)] + fields[1:]))
            counts[out][cid] += 1
        (dst / out / "labels" / f"{stem}.txt").write_text("\n".join(kept) + ("\n" if kept else ""), encoding="utf-8")
        link = dst / out / "images" / img.name
        if use_links:
            try:
                os.symlink(img.resolve(), link)
                continue
            except OSError:
                use_links = False                 # e.g. Windows without developer mode
        shutil.copy2(img, link)
    (dst / "splits.csv").write_text("stem,split\n" + "".join(f"{k},{v}\n" for k, v in sorted(split_of.items())))
    (dst / ".prepared").write_text("ok\n")
    if dropped:
        print("Dropped 'other corruption' boxes:", sum(dropped.values()))
    for s in SPLITS:
        n = sum(1 for v in split_of.values() if v == s)
        print(f"{s:>5}: {n:>6} images | boxes " + ", ".join(f"{CLASS_NAMES[c]} {counts[s][c]}" for c in CLASS_NAMES))


if not (DATA / ".prepared").exists():
    shutil.rmtree(DATA, ignore_errors=True)       # a half-finished copy from an interrupted run
    source = get_raw_dataset()
    print("Dataset source:", source)
    prepare_dataset(source, DATA)
else:
    print("Prepared data already at", DATA)

DATA_YAML.write_text(
    f"path: {DATA.as_posix()}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n"
    + "".join(f"  {k}: {v}\n" for k, v in CLASS_NAMES.items()), encoding="utf-8")
print(DATA_YAML.read_text())

## Metric helpers
F1 @ IoU 0.5 (the RDD challenge metric) over a sweep of confidence thresholds, plus AP@0.5. Predictions
are greedily matched to ground truth per class, by descending confidence.

In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)


def box_iou_np(a, b):
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)


class DetectionEvaluator:
    """Collects predictions once, then gives P / R / F1 at any confidence threshold and AP@0.5."""

    def __init__(self, nc, iou_thr=0.5):
        self.nc, self.iou_thr = nc, iou_thr
        self.n_gt = np.zeros(nc, dtype=int)
        self.scores, self.cls, self.tp = [], [], []

    def add(self, pred_boxes, pred_scores, pred_cls, gt_boxes, gt_cls):
        pred_boxes = np.asarray(pred_boxes, float).reshape(-1, 4)
        pred_scores = np.asarray(pred_scores, float).reshape(-1)
        pred_cls = np.asarray(pred_cls).astype(int).reshape(-1)
        gt_boxes = np.asarray(gt_boxes, float).reshape(-1, 4)
        gt_cls = np.asarray(gt_cls).astype(int).reshape(-1)
        for c in range(self.nc):
            g = gt_boxes[gt_cls == c]
            self.n_gt[c] += len(g)
            m = pred_cls == c
            if not m.any():
                continue
            order = np.argsort(-pred_scores[m])
            b, s = pred_boxes[m][order], pred_scores[m][order]
            tp = np.zeros(len(b), dtype=bool)
            if len(g):
                ious, matched = box_iou_np(b, g), np.zeros(len(g), dtype=bool)
                for i in range(len(b)):
                    cand = np.where(matched, -1.0, ious[i])
                    j = int(cand.argmax())
                    if cand[j] >= self.iou_thr:
                        matched[j] = tp[i] = True
                        if matched.all():
                            break
            self.scores.append(s)
            self.cls.append(np.full(len(b), c))
            self.tp.append(tp)

    def _arrays(self):
        if not self.scores:
            return np.zeros(0), np.zeros(0, dtype=int), np.zeros(0, dtype=bool)
        return np.concatenate(self.scores), np.concatenate(self.cls), np.concatenate(self.tp)

    @staticmethod
    def _prf(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        return p, r, (2 * p * r / (p + r) if p + r else 0.0)

    def at_threshold(self, conf):
        scores, cls, tp = self._arrays()
        keep = scores >= conf
        tp_c = np.bincount(cls[keep & tp], minlength=self.nc)
        fp_c = np.bincount(cls[keep & ~tp], minlength=self.nc)
        fn_c = self.n_gt - tp_c
        per_class = [dict(zip(("precision", "recall", "f1"), self._prf(tp_c[c], fp_c[c], fn_c[c])))
                     for c in range(self.nc)]
        p, r, f1 = self._prf(tp_c.sum(), fp_c.sum(), fn_c.sum())
        return {"conf": float(conf), "precision": p, "recall": r, "f1": f1, "per_class": per_class}

    def best(self):
        rows = [self.at_threshold(t) for t in THRESHOLDS]
        return max(rows, key=lambda r: r["f1"]), rows

    def ap50(self):
        """Per-class AP with 101-point interpolation (COCO style)."""
        scores, cls, tp = self._arrays()
        ap = np.zeros(self.nc)
        for c in range(self.nc):
            if self.n_gt[c] == 0:
                continue
            hits = tp[cls == c][np.argsort(-scores[cls == c], kind="stable")]
            tpc, fpc = np.cumsum(hits), np.cumsum(~hits)
            recall = tpc / self.n_gt[c]
            precision = tpc / np.maximum(tpc + fpc, 1)
            envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
            idx = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
            ap[c] = np.mean([envelope[i] if i < len(envelope) else 0.0 for i in idx])
        return ap


def read_yolo_labels(path):
    """YOLO label file -> (class_ids, normalised xyxy boxes)."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    arr = np.loadtxt(path, ndmin=2)
    cx, cy, w, h = arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
    return arr[:, 0].astype(int), np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)


def save_result(row, details):
    """Append one row to results/metrics.csv and save the full details as JSON."""
    import pandas as pd
    csv_path = RESULTS / "metrics.csv"
    df = pd.DataFrame([row])
    if csv_path.exists() and csv_path.stat().st_size:
        df = pd.concat([pd.read_csv(csv_path), df], ignore_index=True)
    df.to_csv(csv_path, index=False)
    (RESULTS / f"{row['run']}_{row['split']}.json").write_text(json.dumps(details, indent=2, default=float))

## Train
Resumes automatically from the last checkpoint on Drive. `MAX_HOURS` stops training cleanly and keeps a
resumable `resume.pt`, so *Run all* in a new session continues where it stopped.

In [ ]:
if "DATA_YAML" not in globals() or not DATA_YAML.exists():
    raise RuntimeError("This Colab runtime was reset (idle timeout or the daily GPU limit), so installed "
                       "packages and the prepared data are gone. Use Runtime → Run all: training resumes "
                       "from the last checkpoint saved on Google Drive.")

from ultralytics import RTDETR, YOLO

ModelClass = RTDETR if "rtdetr" in WEIGHTS.lower() else YOLO
run_dir = RUNS / RUN_NAME
weights_dir = run_dir / "weights"


def resumable(pt):
    """True if the checkpoint still has its optimizer state (Ultralytics strips it when training ends)."""
    if not Path(pt).is_file():
        return False
    ckpt = torch.load(pt, map_location="cpu", weights_only=False)
    return ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) != -1


def add_time_budget(model):
    """Stop before MAX_HOURS, first copying last.pt (still resumable at that point) to resume.pt."""
    start, state = time.time(), {"paused": False}

    def on_fit_epoch_end(trainer):
        if not MAX_HOURS or state["paused"] or trainer.stop:
            return
        elapsed = time.time() - start
        per_epoch = elapsed / max(trainer.epoch + 1 - trainer.start_epoch, 1)
        if elapsed + 1.2 * per_epoch > MAX_HOURS * 3600:
            shutil.copy2(trainer.last, trainer.wdir / "resume.pt")
            trainer.stop = state["paused"] = True
            print(f"\nTime budget reached after epoch {trainer.epoch + 1}/{trainer.epochs}. Run all again to continue.")

    def on_train_end(trainer):
        if state["paused"]:
            (Path(trainer.save_dir) / "PAUSED").write_text("paused\n")
        else:
            (Path(trainer.save_dir) / "PAUSED").unlink(missing_ok=True)
            (trainer.wdir / "resume.pt").unlink(missing_ok=True)

    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    model.add_callback("on_train_end", on_train_end)


if (run_dir / "DONE").exists():
    print("Training already finished:", run_dir)
else:
    ckpt = next((p for p in (weights_dir / "last.pt", weights_dir / "resume.pt") if resumable(p)), None)
    if ckpt:
        print("Resuming from", ckpt)
        model = ModelClass(str(ckpt))
        add_time_budget(model)
        model.train(resume=True)
    else:
        model = ModelClass(WEIGHTS)
        add_time_budget(model)
        model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
                    device=DEVICE, workers=WORKERS, cache=CACHE, patience=30, project=str(RUNS), name=RUN_NAME,
                    exist_ok=True)
    if (weights_dir / "best.pt").exists() and not (run_dir / "PAUSED").exists():
        (run_dir / "DONE").write_text(time.strftime("%Y-%m-%d %H:%M:%S\n"))

FINISHED = (run_dir / "DONE").exists()
print("Training finished." if FINISHED else "Paused: run all again (in a new session if needed) to continue.")

## Training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hist = pd.read_csv(run_dir / "results.csv")
hist.columns = hist.columns.str.strip()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hist.plot(x="epoch", y=[c for c in hist.columns if c.startswith("train/")], ax=axes[0], title="Training loss")
hist.plot(x="epoch", y=["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"],
          ax=axes[1], title="Validation metrics")
plt.tight_layout()
plt.show()
hist.tail()

## Evaluate
The confidence threshold is tuned on **val**, and test F1 is reported at exactly that threshold, so test numbers are honest.

In [ ]:
def evaluate_split(weights, split, fixed_conf=None):
    """Ultralytics mAP + our F1@IoU0.5 sweep on one split; appends a row to results/metrics.csv."""
    model = ModelClass(str(weights))
    m = model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=16, conf=0.001, iou=0.6, device=DEVICE,
                  workers=WORKERS, plots=True, project=str(RUNS / "val"), name=f"{RUN_NAME}_{split}",
                  exist_ok=True, verbose=False)
    ap50 = {CLASS_NAMES[int(c)]: float(m.box.ap50[i]) for i, c in enumerate(m.box.ap_class_index)}

    images = sorted(p for p in (DATA / split / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
    evaluator, times = DetectionEvaluator(len(CLASS_NAMES)), []
    # pass the folder, not a list: Ultralytics decodes a whole list into memory up front (out of memory)
    for r in model.predict(source=str(DATA / split / "images"), imgsz=IMGSZ, conf=0.001, iou=0.6, device=DEVICE,
                           batch=16, stream=True, verbose=False):
        gt_cls, gt_boxes = read_yolo_labels(DATA / split / "labels" / f"{Path(r.path).stem}.txt")
        evaluator.add(r.boxes.xyxyn.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy(),
                      gt_boxes, gt_cls)
        times.append(sum(r.speed.values()))
    best, sweep = evaluator.best()
    fixed = evaluator.at_threshold(fixed_conf) if fixed_conf is not None else None
    ms = float(np.mean(times[5:] if len(times) > 10 else times))

    print(f"\n=== {RUN_NAME} | {split} | {len(images)} images ===")
    print(f"mAP@0.5 = {m.box.map50:.4f} | mAP@0.5:0.95 = {m.box.map:.4f}")
    print(f"Best F1@IoU0.5 = {best['f1']:.4f} at conf {best['conf']:.2f} (P {best['precision']:.3f}, R {best['recall']:.3f})")
    if fixed:
        print(f"F1 at the val-tuned conf {fixed['conf']:.2f} = {fixed['f1']:.4f} "
              f"(P {fixed['precision']:.3f}, R {fixed['recall']:.3f})")
    print(f"Speed: {ms:.1f} ms/img ({1000 / ms:.1f} FPS)")
    for c, name in CLASS_NAMES.items():
        print(f"  {name}: AP50 {ap50.get(name, 0):.3f} | best-conf F1 {best['per_class'][c]['f1']:.3f}")

    row = {"run": RUN_NAME, "split": split, "imgsz": IMGSZ, "precision": best["precision"], "recall": best["recall"],
           "f1": best["f1"], "f1_conf": best["conf"], "map50": float(m.box.map50), "map50_95": float(m.box.map),
           "ms_per_img": ms, "fps": 1000 / ms,
           "fixed_conf": fixed["conf"] if fixed else None, "f1_at_fixed_conf": fixed["f1"] if fixed else None,
           **{f"ap50_{n}": ap50.get(n) for n in CLASS_NAMES.values()}}
    save_result(row, {**row, "best": best, "fixed": fixed, "sweep": sweep})
    return row


if FINISHED:
    best_weights = weights_dir / "best.pt"
    val_row = evaluate_split(best_weights, "val")
    test_row = evaluate_split(best_weights, "test", fixed_conf=val_row["f1_conf"])   # threshold tuned on val only
else:
    print("Training is not finished yet; evaluation runs after the last epoch.")

## Results of all models
Each notebook adds its rows to the same `results/metrics.csv` on Drive, so this table fills up as you train the 4 models.

In [ ]:
import pandas as pd

table = pd.read_csv(RESULTS / "metrics.csv") if (RESULTS / "metrics.csv").exists() else pd.DataFrame()
cols = ["run", "split", "precision", "recall", "f1", "f1_conf", "f1_at_fixed_conf", "map50", "map50_95", "fps"]
print(f"All results so far ({RESULTS / 'metrics.csv'}). Report test F1 at the val-tuned threshold "
      "(f1_at_fixed_conf), not test's own best F1.")
table[[c for c in cols if c in table.columns]].round(4) if len(table) else "No results yet."